In [ ]:
# OpenCode / OpenAI-compatible notebook setup.
# Install once in the notebook's Python environment: pip install openai python-dotenv pyyaml
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import os as _nb_os

from pathlib import Path as _NotebookPath

from dotenv import load_dotenv as _notebook_load_dotenv
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Paths
#
CHAPTER = "s12_cron_scheduler"

PROJECT_ROOT = next(
    (
        directory
        for start in [_NotebookPath.cwd(), *_NotebookPath.cwd().parents]
        for directory in [start]
        if ((directory / CHAPTER / "code.py").is_file())
    ),
    None,
)

if (PROJECT_ROOT is None):

    raise (FileNotFoundError(
        f"Start Jupyter from the project root or {CHAPTER}/ "
        "so code.py can be located"
    ))
#

_notebook_load_dotenv(PROJECT_ROOT / ".env", override=True)
_nb_os.chdir(PROJECT_ROOT)

__file__ = str(PROJECT_ROOT / CHAPTER / "code.py")
#
###################################################################################################
###################################################################################################
################################################################################################### Interactive Entry Point
#
# Run All must not start a blocking CLI. Enable this explicitly if needed.
RUN_AGENT_INTERACTIVELY = False
#
###################################################################################################
###################################################################################################
################################################################################################### OpenCode Configuration
#
# The existing ANTHROPIC_* environment names remain supported.
_notebook_key   = (_nb_os.getenv("OPENCODE_API_KEY")  or  _nb_os.getenv("ANTHROPIC_API_KEY"))
_notebook_model = (_nb_os.getenv("OPENCODE_MODEL")  or  _nb_os.getenv("MODEL_ID"))
_notebook_url   = (
    _nb_os.getenv("OPENCODE_BASE_URL")
    or  _nb_os.getenv("ANTHROPIC_BASE_URL")
    or  "https://opencode.ai/inference/openai/v1"
).rstrip("/")

if (_notebook_url.endswith("/chat/completions")):

    _notebook_url = _notebook_url[:-len("/chat/completions")]
#

if (not _notebook_key):

    raise (RuntimeError(
        "Set OPENCODE_API_KEY (or ANTHROPIC_API_KEY) in the root .env"
    ))
#

if (not _notebook_model):

    raise (RuntimeError(
        "Set MODEL_ID (or OPENCODE_MODEL) in the root .env"
    ))
#

_nb_os.environ["OPENCODE_API_KEY"]   = _notebook_key
_nb_os.environ["OPENCODE_BASE_URL"]  = _notebook_url
_nb_os.environ["ANTHROPIC_BASE_URL"] = _notebook_url
_nb_os.environ["MODEL_ID"]          = _notebook_model

print(f"{CHAPTER}: OpenCode via OpenAI-compatible API, model={_notebook_model}")
#


In [ ]:
# Bridge the lesson's Anthropic-style message interface to OpenCode's OpenAI API.
# Legacy chapter loaders import "anthropic" dynamically; this mapping serves those
# imports inside the current notebook kernel without changing the .py files.
#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import json  as _compat_json
import sys   as _compat_sys
import types as _compat_types

from openai import OpenAI
import uuid
#
###################################################################################################
###################################################################################################
################################################################################################### Content Conversion
#
def _compat_field(block:object, key:str, default=None):

    """
        Read a response field from either a plain dictionary or provider object.

        The chapter code consumes both forms; this helper keeps those access
        patterns equivalent without mutating the original response.
    """

    return ((block.get(key, default))
            if (isinstance(block, dict))
            else (getattr(block, key, default)))
#


def _compat_text(value:object) -> str:

    """
        Convert Anthropic-style content blocks to Chat Completions text.

        Exclude tool-use blocks: their identifiers and arguments are translated
        separately so that subsequent tool results can be matched correctly.
    """

    if (isinstance(value, str)):

        return (value)
    #
    if (value is None):

        return ("")
    #
    if (isinstance(value, list)):

        return ("\n".join(
            _compat_text(_compat_field(item, "text", _compat_field(item, "content", "")))
            for item in value
            if (_compat_field(item, "type") != "tool_use")
        ))
    #
    if (isinstance(value, dict)):

        return (_compat_text(value.get(
            "text", value.get("content", _compat_json.dumps(value))
        )))
    #
    return (str(value))
#


def _compat_history(messages:list, system:str|list|None=None) -> list:

    """
        Translate chapter conversation history for OpenAI Chat Completions.

        Preserve every assistant tool-call ID and pair it with the corresponding
        tool response. The system prompt is inserted before conversational turns.
    """

    result = []

    if (system):

        result.append({"role": "system", "content": _compat_text(system)})
    #

    for message in messages:

        role    = _compat_field(message, "role")
        content = _compat_field(message, "content", "")

        if (role == "assistant"):

            if (not isinstance(content, list)):

                result.append({"role": "assistant", "content": _compat_text(content)})
                continue
            #

            text_blocks = []
            tool_calls  = []

            for block in content:

                kind = _compat_field(block, "type")

                if (kind == "tool_use"):

                    arguments = _compat_field(block, "input", {})

                    tool_calls.append({
                        "id"       : _compat_field(block, "id"),
                        "type"     : "function",
                        "function" : {
                            "name"      : _compat_field(block, "name"),
                            "arguments" : _compat_json.dumps(arguments, ensure_ascii=False),
                        },
                    })
                #
                elif (kind == "text"):

                    text_blocks.append(_compat_field(block, "text", ""))
                #
            #

            converted = {"role": "assistant", "content": "\n".join(text_blocks) or None}

            if (tool_calls):

                converted["tool_calls"] = tool_calls
            #
            if (converted["content"] is None  and  not tool_calls):

                converted["content"] = ""
            #

            result.append(converted)
        #
        elif (role == "user"  and  isinstance(content, list)):

            user_text    = []
            tool_results = []

            for block in content:

                if (_compat_field(block, "type") == "tool_result"):

                    tool_results.append({
                        "role"         : "tool",
                        "tool_call_id" : _compat_field(block, "tool_use_id"),
                        "content"      : _compat_text(_compat_field(block, "content", "")),
                    })
                #
                else:

                    user_text.append(_compat_text(block))
                #
            #

            result.extend(tool_results)

            if (user_text):

                result.append({"role": "user", "content": "\n".join(user_text)})
            #
        #
        else:

            result.append({"role": role, "content": _compat_text(content)})
        #
    #

    return (result)
#
###################################################################################################
###################################################################################################
################################################################################################### OpenAI Message Adapter
#
class _NotebookMessages:

    """
        Expose messages.create on an OpenAI-compatible chat-completions client.

        Construct function-tool definitions, convert responses to the chapter's
        expected block objects, and translate token counts and stop reasons.
    """

    def __init__(self, client) -> None:

        """
            Keep the provider client used by subsequent model requests.
        """

        self._client = client
    #
    #################################################################  Public API
    #
    def create(
            self,
            *,
            model      : str,
            messages   : list,
            system     : str|list|None = None,
            tools      : list|None     = None,
            max_tokens : int|None      = None,
            **kwargs,
        ) -> object:

        """
            Send an OpenAI-compatible request and return Anthropic-shaped blocks.

            Tool requests remain associated with their call identifiers.
            The translation preserves the original lesson's model/tool loop.
        """

        request = {"model": model, "messages": _compat_history(messages, system)}

        if (tools):

            request["tools"] = [
                {
                    "type"     : "function",
                    "function" : {
                        "name"        : tool["name"],
                        "description" : tool.get("description", ""),
                        "parameters"  : tool.get(
                            "input_schema", {"type": "object", "properties": {}}
                        ),
                    },
                }
                for tool in tools
            ]
        #
        if (max_tokens is not None):

            request["max_tokens"] = max_tokens
        #

        response = self._client.chat.completions.create(**request)
        choice   = response.choices[0]
        reply    = choice.message
        blocks   = []

        if (reply.content):

            blocks.append(_compat_types.SimpleNamespace(
                type = "text",
                text = reply.content,
            ))
        #

        for call in (reply.tool_calls or []):

            try:

                arguments = _compat_json.loads(call.function.arguments or "{}")

                if (not isinstance(arguments, dict)):

                    arguments = {}
                #
            #
            except (ValueError, TypeError):

                arguments = {}
            #

            blocks.append(_compat_types.SimpleNamespace(
                type  = "tool_use",
                id    = call.id,
                name  = call.function.name,
                input = arguments,
            ))
        #

        usage       = getattr(response, "usage", None)
        stop_reason = (
            ("max_tokens") if (choice.finish_reason == "length")
            else (("tool_use") if (reply.tool_calls) else ("end_turn"))
        )

        return (_compat_types.SimpleNamespace(
            content     = blocks,
            stop_reason = stop_reason,
            usage       = _compat_types.SimpleNamespace(
                input_tokens  = getattr(usage, "prompt_tokens", 0)  or  0,
                output_tokens = getattr(usage, "completion_tokens", 0)  or  0,
            ),
        ))
    #
#
###################################################################################################
###################################################################################################
################################################################################################### Notebook Client
#
class _NotebookAnthropic:

    """
        Provide a legacy chapter client backed by the OpenCode endpoint.

        The name is retained for import compatibility; requests are always sent
        through the OpenAI Python SDK and use the notebook's configured API key.
    """

    def __init__(self, base_url:str|None=None, **kwargs) -> None:

        """
            Initialize the OpenAI client and expose the message adapter.
        """

        url = (_nb_os.getenv("OPENCODE_BASE_URL")  or  base_url  or  _notebook_url)

        self._client = _OpenAI(api_key=_notebook_key, base_url=url)
        self.messages = _NotebookMessages(self._client)
    #
#


_anthropic_compat           = _compat_types.ModuleType("anthropic")
_anthropic_compat.Anthropic = _NotebookAnthropic
_compat_sys.modules["anthropic"] = _anthropic_compat
#


In [ ]:
#!/usr/bin/env python3
"""
s12_cron_scheduler.py - Cron Scheduler

    +--------------------------+   09:00   +-----------------------+
    | 0 9 * * *               | --------> | [Scheduled] run tests |
    | prompt: "run tests"      |           +-----------+-----------+
    +--------------------------+                       |
          scheduled_jobs                    cron_queue | agent idle
                                                        v
                                                +-------------+
                                                | Agent Loop  |
                                                +-------------+
"""


In [ ]:

#
###################################################################################################
###################################################################################################
################################################################################################### Standard Modules
#
import glob
import json
import os
import re
import secrets
import subprocess
import threading
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path

try:

    import readline

    readline.parse_and_bind("set bind-tty-special-chars off")
    readline.parse_and_bind("set input-meta on")
    readline.parse_and_bind("set output-meta on")
    readline.parse_and_bind("set convert-meta off")
#
except ImportError:

    pass
#

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)
if (os.getenv("ANTHROPIC_BASE_URL")):

    os.environ.pop("ANTHROPIC_AUTH_TOKEN", None)
#

WORKDIR      = Path.cwd()
DURABLE_PATH = WORKDIR / ".scheduled_tasks.json"
client       = Anthropic(base_url=os.getenv("ANTHROPIC_BASE_URL"))
MODEL        = os.environ["MODEL_ID"]

ENVIRONMENT_PROMPT = (
    ("Windows: the bash tool runs through cmd.exe; use cmd.exe syntax, not Unix "
    "Bash or PowerShell syntax, and prefer dedicated file tools for file operations")
    if (os.name == "nt")
    else ("Unix-like: the bash tool runs the system shell")
)
SYSTEM             = (
    f"You are a coding agent at {WORKDIR}. Environment: {ENVIRONMENT_PROMPT}. "
    "Use tools to solve tasks. "
    "Use schedule_cron for work that should start at a future local time."
)


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### From s04: tool implementations
#

def run_bash(command:str) -> str:

    try:

        result = subprocess.run(
            command,
            shell          = True,
            cwd            = WORKDIR,
            capture_output = True,
            text           = True,
            errors         = "replace",
            timeout        = 120,
        )
        output = (result.stdout + result.stderr).strip()
        if (result.returncode != 0):

            return (f"Error: command exited with status {result.returncode}\n{output}")
        #
        return ((output[:50000]) if (output) else ("(no output)"))
    #
    except subprocess.TimeoutExpired:

        return ("Error: Timeout (120s)")
    #
#


def run_read(path:str, limit:int|None=None) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        lines     = file_path.read_text(encoding="utf-8").splitlines()
        if (limit  and  limit < len(lines)):

            lines = lines[:limit] + [f"... ({len(lines) - limit} more lines)"]
        #
        return ("\n".join(lines))
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


def run_write(path:str, content:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        file_path.parent.mkdir(parents=True, exist_ok=True)
        file_path.write_text(content, encoding="utf-8")
        return (f"Wrote {len(content)} bytes to {path}")
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


def run_edit(path:str, old_text:str, new_text:str) -> str:

    try:

        file_path = (WORKDIR / path).resolve()
        text      = file_path.read_text(encoding="utf-8")
        if (old_text not in text):

            return (f"Error: text not found in {path}")
        #
        file_path.write_text(text.replace(old_text, new_text, 1), encoding="utf-8")
        return (f"Edited {path}")
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


def run_glob(pattern:str) -> str:

    try:

        matches = sorted({
            match
            for match in glob.glob(pattern, root_dir=WORKDIR, recursive=True)
            if (WORKDIR / match).resolve().is_relative_to(WORKDIR)
        })
        shown   = matches[:200]
        if (len(matches) > 200):

            shown.append("... (more matches omitted; narrow the pattern)")
        #
        return (("\n".join(shown)) if (shown) else ("(no matches)"))
    #
    except Exception as error:

        return (f"Error: {error}")
    #
#


TOOLS = [
    {
        "name"         : "bash",
        "description"  : "Run a shell command.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"command": {"type": "string"}},
            "required"   : ["command"],
        },
    },
    {
        "name"         : "read_file",
        "description"  : "Read file contents.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"  : {"type": "string"},
                "limit" : {"type": "integer"},
            },
            "required"   : ["path"],
        },
    },
    {
        "name"         : "write_file",
        "description"  : "Write content to a file.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"    : {"type": "string"},
                "content" : {"type": "string"},
            },
            "required"   : ["path", "content"],
        },
    },
    {
        "name"         : "edit_file",
        "description"  : "Replace exact text in a file once.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "path"     : {"type": "string"},
                "old_text" : {"type": "string"},
                "new_text" : {"type": "string"},
            },
            "required"   : ["path", "old_text", "new_text"],
        },
    },
    {
        "name"         : "glob",
        "description"  : "Find files matching a glob pattern; ** matches recursively.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"pattern": {"type": "string"}},
            "required"   : ["pattern"],
        },
    },
]

TOOL_HANDLERS = {
    "bash"       : run_bash,
    "read_file"  : run_read,
    "write_file" : run_write,
    "edit_file"  : run_edit,
    "glob"       : run_glob,
}


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### From s04: hooks and permission checks
#

HOOKS = {"UserPromptSubmit": [], "PreToolUse": [], "PostToolUse": [], "Stop": []}


def register_hook(event:str, callback):

    HOOKS[event].append(callback)
#


def trigger_hooks(event:str, *args):

    for callback in HOOKS[event]:

        result = callback(*args)
        if (result is not None):

            return (result)
        #
    #
    return (None)
#


DENY_LIST                = ["rm -rf /", "sudo", "shutdown", "reboot", "mkfs", "dd if="]
DESTRUCTIVE_COMMAND_WORD = re.compile(
    r"(?i)(?:^|[;&|()\n])\s*(?:rm|del)(?=\s|$|[;&|()])"
)
DESTRUCTIVE              = ["rm ", "> /etc/", "chmod 777"]


def contains_destructive_command(command:str) -> bool:

    return (bool(DESTRUCTIVE_COMMAND_WORD.search(command)))
#


def request_permission(block, reason:str) -> str|None:

    if (threading.current_thread() is not threading.main_thread()):

        return ("Permission denied: scheduled turns cannot request interactive approval")
    #

    print(f"\n\033[33m[permission] {reason}\033[0m")
    print(f"   Tool: {block.name}({block.input})")
    choice = input("   Allow? [y/N] ").strip().lower()
    if (choice not in ("y", "yes")):

        return ("Permission denied by user")
    #
    return (None)
#


def permission_hook(block):

    if (block.name == "bash"):

        command = block.input.get("command", "")
        for pattern in DENY_LIST:

            if (pattern in command):

                print(f"\n\033[31m[blocked] '{pattern}'\033[0m")
                return ("Permission denied by deny list")
            #
        #
        if (contains_destructive_command(command)  or  any(
            keyword in command for keyword in DESTRUCTIVE
        )):

            return (request_permission(block, "Potentially destructive command"))
        #
    #

    if (block.name in ("read_file", "write_file", "edit_file")):

        path = block.input.get("path", "")
        if (not (WORKDIR / path).resolve().is_relative_to(WORKDIR)):

            return (request_permission(block, "Access outside workspace"))
        #
    #
    return (None)
#


def log_hook(block):

    preview = str(list(block.input.values())[:2])[:60]
    print(f"\033[90m[HOOK] {block.name}({preview})\033[0m")
    return (None)
#


def large_output_hook(block, output):

    if (len(str(output)) > 100000):

        print(
            f"\033[33m[HOOK] Large output from {block.name}: "
            f"{len(str(output))} chars\033[0m"
        )
    #
    return (None)
#


def context_inject_hook(query:str):

    print(f"\033[90m[HOOK] UserPromptSubmit: working in {WORKDIR}\033[0m")
    return (None)
#


def summary_hook(messages:list):

    tool_count = sum(
        1
        for message in messages
        for block in (
            (message.get("content"))
            if (isinstance(message.get("content"), list))
            else ([])
        )
        if isinstance(block, dict)  and  block.get("type") == "tool_result"
    )
    print(f"\033[90m[HOOK] Stop: session used {tool_count} tool calls\033[0m")
    return (None)
#


register_hook("UserPromptSubmit", context_inject_hook)
register_hook("PreToolUse", permission_hook)
register_hook("PreToolUse", log_hook)
register_hook("PostToolUse", large_output_hook)
register_hook("Stop", summary_hook)


In [ ]:


#
###################################################################################################
###################################################################################################
################################################################################################### New in s12: cron jobs
#

@dataclass
class CronJob:

    id               : str
    cron             : str
    prompt           : str
    recurring        : bool
    durable          : bool
    pending_delivery : bool     = False
    last_fired       : str|None = None
#


scheduled_jobs: dict[str, CronJob] = {}
cron_queue: list[CronJob] = []
cron_lock = threading.RLock()


def _cron_field_matches(field:str, value:int) -> bool:

    if (field == "*"):

        return (True)
    #
    if (field.startswith("*/")):

        return (value % int(field[2:]) == 0)
    #
    if ("," in field):

        return (any(_cron_field_matches(part.strip(), value)
                   for part in field.split(",")))
    #
    if ("-" in field):

        start, end = field.split("-", 1)
        return (int(start) <= value <= int(end))
    #
    return (value == int(field))
#


def cron_matches(cron_expr:str, moment:datetime) -> bool:

    fields = cron_expr.strip().split()
    if (len(fields) != 5):

        return (False)
    #

    minute, hour, day, month, weekday = fields
    cron_weekday                      = (moment.weekday() + 1) % 7
    if (not (
        _cron_field_matches(minute, moment.minute)
        and  _cron_field_matches(hour, moment.hour)
        and  _cron_field_matches(month, moment.month)
    )):

        return (False)
    #

    day_matches     = _cron_field_matches(day, moment.day)
    weekday_matches = _cron_field_matches(weekday, cron_weekday)
    if (day == "*"  and  weekday == "*"):

        return (True)
    #
    if (day == "*"):

        return (weekday_matches)
    #
    if (weekday == "*"):

        return (day_matches)
    #
    return (day_matches  or  weekday_matches)
#


def _validate_cron_field(field:str, minimum:int, maximum:int) -> str|None:

    if (field == "*"):

        return (None)
    #
    if (field.startswith("*/")):

        step = field[2:]
        if (not step.isdigit()  or  int(step) <= 0):

            return (f"Invalid step: {field}")
        #
        return (None)
    #
    if ("," in field):

        for part in field.split(","):

            error = _validate_cron_field(part.strip(), minimum, maximum)
            if (error):

                return (error)
            #
        #
        return (None)
    #
    if ("-" in field):

        start, end = field.split("-", 1)
        if (not start.isdigit()  or  not end.isdigit()):

            return (f"Invalid range: {field}")
        #
        start_value, end_value = int(start), int(end)
        if (start_value > end_value):

            return (f"Range start is greater than end: {field}")
        #
        if (start_value < minimum  or  end_value > maximum):

            return (f"Range {field} is outside [{minimum}-{maximum}]")
        #
        return (None)
    #
    if (not field.isdigit()):

        return (f"Invalid field: {field}")
    #
    value = int(field)
    if (value < minimum  or  value > maximum):

        return (f"Value {value} is outside [{minimum}-{maximum}]")
    #
    return (None)
#


def validate_cron(cron_expr:str) -> str|None:

    fields = cron_expr.strip().split()
    if (len(fields) != 5):

        return (f"Expected 5 fields, got {len(fields)}")
    #

    field_rules = [
        ("minute", 0, 59),
        ("hour", 0, 23),
        ("day-of-month", 1, 31),
        ("month", 1, 12),
        ("day-of-week", 0, 6),
    ]
    for field, (name, minimum, maximum) in zip(fields, field_rules):

        error = _validate_cron_field(field, minimum, maximum)
        if (error):

            return (f"{name}: {error}")
        #
    #
    return (None)
#


def save_durable_jobs():

    with cron_lock:

        payload   = [
            asdict(job)
            for job in scheduled_jobs.values()
            if job.durable
        ]
        temporary = DURABLE_PATH.with_name(
            f"{DURABLE_PATH.name}.{os.getpid()}.{threading.get_ident()}.tmp"
        )
        try:

            temporary.write_text(json.dumps(payload, indent=2), encoding="utf-8")
            os.replace(temporary, DURABLE_PATH)
        #
        finally:

            temporary.unlink(missing_ok=True)
        #
    #
#




In [ ]:
def load_durable_jobs():

    if (not DURABLE_PATH.exists()):

        return
    #
    try:

        payload = json.loads(DURABLE_PATH.read_text(encoding="utf-8"))
        if (not isinstance(payload, list)):

            raise (ValueError("expected a JSON list"))
        #
    #
    except (OSError, json.JSONDecodeError, ValueError) as error:

        print(f"  [cron] could not load {DURABLE_PATH.name}: {error}")
        return
    #

    loaded = 0
    with cron_lock:

        for item in payload:

            try:

                job   = CronJob(**item)
                error = validate_cron(job.cron)
                if (error):

                    raise (ValueError(error))
                #
                if (not job.id.startswith("cron_")):

                    raise (ValueError("invalid job ID"))
                #
                if (not job.prompt.strip()):

                    raise (ValueError("prompt cannot be empty"))
                #
            #
            except (TypeError, ValueError) as error:

                print(f"  [cron] skipped invalid saved job: {error}")
                continue
            #
            scheduled_jobs[job.id] = job
            if (job.pending_delivery):

                cron_queue.append(job)
            #
            loaded += 1
        #
    #
    if (loaded):

        print(f"  [cron] loaded {loaded} durable job(s)")
    #
#


def new_cron_id() -> str:

    for _ in range(100):

        job_id = f"cron_{secrets.token_hex(4)}"
        if (job_id not in scheduled_jobs):

            return (job_id)
        #
    #
    raise (RuntimeError("Could not allocate a cron job ID"))
#


def schedule_job(cron:str, prompt:str, recurring:bool=True, durable:bool=True) -> CronJob|str:

    error = validate_cron(cron)
    if (error):

        return (error)
    #
    if (not prompt.strip()):

        return ("Prompt cannot be empty")
    #

    with cron_lock:

        job                    = CronJob(
            id        = new_cron_id(),
            cron      = cron,
            prompt    = prompt,
            recurring = recurring,
            durable   = durable,
        )
        scheduled_jobs[job.id] = job
        try:

            if (durable):

                save_durable_jobs()
            #
        #
        except Exception:

            scheduled_jobs.pop(job.id, None)
            raise
        #
    #
    print(f"  [cron] scheduled {job.id}: {cron} -> {prompt[:60]}")
    return (job)
#


def cancel_job(job_id:str) -> str:

    with cron_lock:

        job = scheduled_jobs.get(job_id)
        if (job is None):

            return (f"Job {job_id} not found")
        #

        previous_queue = list(cron_queue)
        scheduled_jobs.pop(job_id)
        cron_queue[:] = [queued for queued in cron_queue if queued.id != job_id]
        try:

            if (job.durable):

                save_durable_jobs()
            #
        #
        except Exception:

            scheduled_jobs[job_id] = job
            cron_queue[:]          = previous_queue
            raise
        #
    #
    print(f"  [cron] cancelled {job_id}")
    return (f"Cancelled {job_id}")
#


def _enqueue_due_job(job:CronJob, minute_marker:str|None=None):

    old_pending          = job.pending_delivery
    old_last_fired       = job.last_fired
    job.pending_delivery = True
    if (minute_marker is not None):

        job.last_fired = minute_marker
    #
    try:

        if (job.durable):

            save_durable_jobs()
        #
    #
    except Exception:

        job.pending_delivery = old_pending
        job.last_fired       = old_last_fired
        raise
    #
    cron_queue.append(job)
#


def poll_due_jobs(moment:datetime):

    minute_marker = moment.strftime("%Y-%m-%d %H:%M")
    with cron_lock:

        for job in list(scheduled_jobs.values()):

            try:

                if (job.pending_delivery  or  job.last_fired == minute_marker):

                    continue
                #
                if (cron_matches(job.cron, moment)):

                    _enqueue_due_job(job, minute_marker)
                    print(f"  [cron] due {job.id}: {job.prompt[:60]}")
                #
            #
            except Exception as error:

                print(f"  [cron] could not enqueue {job.id}: {error}")
            #
        #
    #
#




In [ ]:
def consume_cron_queue() -> list[CronJob]:

    with cron_lock:

        jobs = list(cron_queue)
        cron_queue.clear()
    #
    return (jobs)
#


def acknowledge_cron_jobs(jobs:list[CronJob]):

    changed : list[tuple[CronJob, bool]] = []
    removed : list[CronJob]              = []
    with cron_lock:

        for delivered in jobs:

            current = scheduled_jobs.get(delivered.id)
            if (current is None):

                continue
            #
            changed.append((current, current.pending_delivery))
            if (current.recurring):

                current.pending_delivery = False
            #
            else:

                removed.append(current)
                scheduled_jobs.pop(current.id)
            #
        #

        try:

            if (any(job.durable for job, _ in changed)):

                save_durable_jobs()
            #
        #
        except Exception:

            for job in removed:

                scheduled_jobs[job.id] = job
            #
            for job, pending in changed:

                job.pending_delivery = pending
            #
            queued_ids = {job.id for job in cron_queue}
            for job, _ in changed:

                if (job.id not in queued_ids):

                    cron_queue.append(job)
                #
            #
            raise
        #
    #
#


def restore_cron_jobs(jobs:list[CronJob]):

    with cron_lock:

        queued_ids = {job.id for job in cron_queue}
        for delivered in jobs:

            current = scheduled_jobs.get(delivered.id)
            if (current is None):

                continue
            #
            current.pending_delivery = True
            if (current.id not in queued_ids):

                cron_queue.append(current)
                queued_ids.add(current.id)
            #
        #
    #
#


def has_cron_queue() -> bool:

    with cron_lock:

        return (bool(cron_queue))
    #
#


def run_schedule_cron(cron:str, prompt:str, recurring:bool=True, durable:bool=True) -> str:

    result = schedule_job(cron, prompt, recurring, durable)
    if (isinstance(result, str)):

        return (f"Error: {result}")
    #
    return (f"Scheduled {result.id}: {cron} -> {prompt}")
#


def run_list_crons() -> str:

    with cron_lock:

        jobs = list(scheduled_jobs.values())
    #
    if (not jobs):

        return ("No cron jobs.")
    #

    lines = []
    for job in jobs:

        frequency = ("recurring") if (job.recurring) else ("one-shot")
        storage   = ("durable") if (job.durable) else ("session")
        lines.append(
            f"{job.id}: {job.cron} -> {job.prompt[:60]} "
            f"[{frequency}, {storage}]"
        )
    #
    return ("\n".join(lines))
#


def run_cancel_cron(job_id:str) -> str:

    return (cancel_job(job_id))
#


TOOLS.extend([
    {
        "name"         : "schedule_cron",
        "description"  : "Schedule a prompt with a 5-field cron expression.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {
                "cron"      : {"type": "string"},
                "prompt"    : {"type": "string"},
                "recurring" : {"type": "boolean"},
                "durable"   : {"type": "boolean"},
            },
            "required"   : ["cron", "prompt"],
        },
    },
    {
        "name"         : "list_crons",
        "description"  : "List scheduled cron jobs.",
        "input_schema" : {"type": "object", "properties": {}, "required": []},
    },
    {
        "name"         : "cancel_cron",
        "description"  : "Cancel a cron job by ID.",
        "input_schema" : {
            "type"       : "object",
            "properties" : {"job_id": {"type": "string"}},
            "required"   : ["job_id"],
        },
    },
])

TOOL_HANDLERS.update({
    "schedule_cron" : run_schedule_cron,
    "list_crons"    : run_list_crons,
    "cancel_cron"   : run_cancel_cron,
})


def execute_tool(block) -> str:

    blocked = trigger_hooks("PreToolUse", block)
    if (blocked is not None):

        return (str(blocked))
    #

    handler = TOOL_HANDLERS.get(block.name)
    try:

        output = (handler(**block.input)) if (handler) else (f"Unknown: {block.name}")
    #
    except Exception as error:

        output = f"Error: {error}"
    #
    trigger_hooks("PostToolUse", block, output)
    return (str(output))


In [ ]:
#


#
###################################################################################################
###################################################################################################
################################################################################################### Scheduler and agent loop
#

RUNTIME_STOP = threading.Event()
runtime_threads: list[threading.Thread] = []
runtime_started = False
runtime_lock = threading.Lock()
agent_lock = threading.Lock()
session_history: list = []


def cron_scheduler_loop(stop_event:threading.Event=RUNTIME_STOP):

    while (not stop_event.wait(1.0)):

        poll_due_jobs(datetime.now())
    #
#


def agent_loop(messages:list, context:dict|None=None):

    fired           = consume_cron_queue()
    scheduled_start = len(messages)
    for job in fired:

        messages.append({"role": "user", "content": f"[Scheduled] {job.prompt}"})
        print(f"  [cron] delivered {job.id}: {job.prompt[:60]}")
    #

    waiting_for_ack = list(fired)
    while (True):

        try:

            response = client.messages.create(
                model      = MODEL,
                system     = SYSTEM,
                messages   = messages,
                tools      = TOOLS,
                max_tokens = 8000,
            )
        #
        except Exception as error:

            if (waiting_for_ack):

                del messages[scheduled_start:]
                restore_cron_jobs(waiting_for_ack)
            #
            print(f"  [error] {type(error).__name__}: {error}")
            return (context)
        #

        messages.append({"role": "assistant", "content": response.content})
        if (waiting_for_ack):

            try:

                acknowledge_cron_jobs(waiting_for_ack)
            #
            except Exception as error:

                print(f"  [cron] acknowledgement failed: {error}")
            #
            waiting_for_ack = []
        #

        tool_calls = [
            block for block in response.content if block.type == "tool_use"
        ]
        if (not tool_calls):

            force = trigger_hooks("Stop", messages)
            if (force):

                messages.append({"role": "user", "content": force})
                continue
            #
            return (context)
        #

        results = []
        for block in tool_calls:

            output = execute_tool(block)
            results.append({
                "type"        : "tool_result",
                "tool_use_id" : block.id,
                "content"     : output,
            })
        #
        messages.append({"role": "user", "content": results})
    #
#


def print_latest_assistant_text(messages:list):

    for message in reversed(messages):

        if (message.get("role") != "assistant"):

            continue
        #
        content = message.get("content", "")
        if (isinstance(content, str)):

            print(content)
        #
        else:

            for block in content:

                if (getattr(block, "type", None) == "text"):

                    print(block.text)
                #
                elif (isinstance(block, dict)  and  block.get("type") == "text"):

                    print(block.get("text", ""))
                #
            #
        #
        return
    #
#


def run_agent_turn_locked(user_query:str|None=None):

    if (user_query is not None):

        trigger_hooks("UserPromptSubmit", user_query)
        session_history.append({"role": "user", "content": user_query})
    #
    agent_loop(session_history)
    print_latest_assistant_text(session_history)
    print()
#


def queue_processor_loop(stop_event:threading.Event=RUNTIME_STOP):

    while (not stop_event.wait(0.2)):

        if (not has_cron_queue()  or  not agent_lock.acquire(blocking=False)):

            continue
        #
        try:

            if (has_cron_queue()):

                run_agent_turn_locked()
            #
        #
        finally:

            agent_lock.release()
        #
    #
#


def start_runtime_threads():

    global runtime_started
    with runtime_lock:

        if (runtime_started):

            return
        #
        load_durable_jobs()
        RUNTIME_STOP.clear()
        runtime_threads.extend([
            threading.Thread(
                target = cron_scheduler_loop,
                name   = "cron-scheduler",
                daemon = True,
            ),
            threading.Thread(
                target = queue_processor_loop,
                name   = "cron-queue-processor",
                daemon = True,
            ),
        ])
        for thread in runtime_threads:

            thread.start()
        #
        runtime_started = True
    #
#




In [ ]:
def stop_runtime_threads():

    global runtime_started
    with runtime_lock:

        if (not runtime_started):

            return
        #
        RUNTIME_STOP.set()
        for thread in runtime_threads:

            thread.join(timeout=1)
        #
        runtime_threads.clear()
        runtime_started = False
    #
#


if (RUN_AGENT_INTERACTIVELY):

    print("s12: Cron Scheduler - run prompts on a local schedule")
    print("Enter a question, press Enter to send. Type q to quit.\n")
    start_runtime_threads()
    try:

        while (True):

            try:

                # \001/\002 tell Readline the ANSI escapes have zero display width.
                query = input("\001\033[36m\002s12 >> \001\033[0m\002")
            #
            except (EOFError, KeyboardInterrupt):

                break
            #
            if (query.strip().lower() in ("q", "exit", "")):

                break
            #
            with agent_lock:

                run_agent_turn_locked(query)
            #
        #
    #
    finally:

        stop_runtime_threads()
    #
#
